In [ ]:
import os
import random
import gc
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold 

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler 

import albumentations as A
from albumentations.pytorch import ToTensorV2

import timm


DATA_DIR = "/kaggle/input/competitions/dog-breed-identification/" 
IMG_SIZE = 384  
BATCH_SIZE = 16 
EPOCHS = 6      
LR = 2e-4          
N_FOLDS = 5         
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "convnext_small.fb_in22k_ft_in1k_384" 

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
print(f"Используемое устройство: {DEVICE}")

df = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))
df['filepath'] = df['id'].apply(lambda x: os.path.join(DATA_DIR, "train", f"{x}.jpg"))

breeds = sorted(df['breed'].unique())
breed_to_idx = {breed: idx for idx, breed in enumerate(breeds)}
df['label'] = df['breed'].map(breed_to_idx)

class DogDataset(Dataset):
    def __init__(self, dataframe, transform=None, is_test=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = np.array(Image.open(row['filepath']).convert("RGB"))

        if self.transform:
            image = self.transform(image=image)['image']

        if self.is_test:
            return image, row['id']
        else:
            return image, torch.tensor(row['label'], dtype=torch.long)


train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(rotate=(-15, 15), translate_percent=(0.0, 0.1), scale=(0.9, 1.1), p=0.5), 
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5), 
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3), 
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])


def train_fold(fold, train_idx, val_idx):
    print(f"\n{'='*20} Обучение Фолда {fold+1}/{N_FOLDS} {'='*20}")
    
    train_data = df.iloc[train_idx]
    val_data = df.iloc[val_idx]
    
    train_dataset = DogDataset(train_data, transform=train_transforms)
    val_dataset = DogDataset(val_data, transform=val_transforms)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(breeds))
    model = model.to(DEVICE)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler() 
    
    best_val_loss = float('inf')
    model_path = f'best_model_fold_{fold}.pth'
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        
        for images, labels in tqdm(train_loader, desc=f"Эпоха {epoch+1}/{EPOCHS} [Обучение]", leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            
            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * images.size(0)
            
        scheduler.step()
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Эпоха {epoch+1}/{EPOCHS} [Валидация]", leave=False):
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                
        val_loss /= len(val_loader.dataset)
        val_acc = correct / len(val_loader.dataset)
        
        print(f"Эпоха {epoch+1}: Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
            print(f"--> Фолд {fold+1}: Модель улучшилась! Сохранено (Loss: {best_val_loss:.4f})")
            
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['label'])):
    train_fold(fold, train_idx, val_idx)

print("\nЗапуск предсказания на тестовом наборе данных...")
test_dir = os.path.join(DATA_DIR, "test")
test_files = os.listdir(test_dir)
test_df = pd.DataFrame({
    'id': [f.split('.')[0] for f in test_files],
    'filepath': [os.path.join(test_dir, f) for f in test_files]
})

test_dataset = DogDataset(test_df, transform=val_transforms, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

models = []
for fold in range(N_FOLDS):
    model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=len(breeds))
    model.load_state_dict(torch.load(f'best_model_fold_{fold}.pth'))
    model.to(DEVICE)
    model.eval()
    models.append(model)

test_preds = []
test_ids = []

with torch.no_grad():
    for images, ids in tqdm(test_loader, desc="Инференс"):
        images = images.to(DEVICE)
        images_flipped = torch.flip(images, dims=[3]) 
        
        batch_preds = torch.zeros((images.size(0), len(breeds))).to(DEVICE)
        
        for model in models:
            outputs_orig = model(images)
            probs_orig = torch.softmax(outputs_orig, dim=1)
            
            outputs_flipped = model(images_flipped)
            probs_flipped = torch.softmax(outputs_flipped, dim=1)
            
            batch_preds += (probs_orig + probs_flipped) / 2.0
            
        
        batch_preds /= N_FOLDS
        
        test_preds.append(batch_preds.cpu().numpy())
        if not test_ids: 
            test_ids.extend(ids)
        else:
            test_ids.extend(ids)

test_preds = np.vstack(test_preds)

submission = pd.DataFrame(test_preds, columns=breeds)
submission.insert(0, 'id', test_ids)
submission.to_csv("submission.csv", index=False)

print("Файл submission.csv успешно сохраненё")